# AlphaGenome Annotation Workflow For Selected Loci

This notebook runs AlphaGenome annotations sequentially over a selected panel of loci. It is intended for workbook-driven runs where API rate limits matter more than parallel cluster throughput.

Use the `annotation_selection.csv` emitted by `6_phase1_dataset_metrics_screening.ipynb`.

In [ ]:
import os
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    candidates = []
    current = (start or Path.cwd()).resolve()
    candidates.extend([current, *current.parents])

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (
            (candidate / 'utils').is_dir()
            and (candidate / 'vignettes').is_dir()
            and (candidate / 'README.md').exists()
        ):
            return candidate

    raise FileNotFoundError('Could not locate eQTL_annotations_for_susine project root')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.locus_manifest import (
    load_annotation_selection,
    load_loci_manifest,
    merge_annotation_selection,
    validate_annotation_selection,
    validate_loci_manifest,
)
from utils.output_layout import get_locus_output_paths
from utils.paths import add_project_root_to_sys_path, configure_runtime_env
from utils.pipeline_alphagenome_batch import (
    estimate_alphagenome_workload,
    load_existing_annotation_result,
    run_alphagenome_annotation,
)

PROJECT_ROOT = add_project_root_to_sys_path(PROJECT_ROOT)
PATHS = configure_runtime_env(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')
print(f'AlphaGenome output dir: {PATHS.output_annotation_alphagenome}')

In [ ]:
# =============================================================================
# USER INPUTS
# =============================================================================

MANIFEST_PATH = PROJECT_ROOT / 'config' / 'loci_manifest_sample_100_per_chrom.csv'
SELECTION_PATH = None
API_KEY_ENV_VAR = 'ALPHAGENOME_API_KEY'
FORCE_RERUN = False
FAIL_FAST = False
LOCUS_IDS = None  # e.g. ['cbx8_chr17_lung']
BATCH_SIZE = 100
MAX_VARIANTS = None
SCORING_MAX_WORKERS = 8
WRITE_BATCH_CHECKPOINTS = True
RETRY_WAIT_SECONDS = 5
SLEEP_SECONDS_BETWEEN_LOCI = 0
VERBOSE_PER_BATCH = True

if SELECTION_PATH is None:
    selection_candidates = sorted(
        (PROJECT_ROOT / 'output' / 'prelim' / 'phase1_metrics_screening_review').glob(
            'representative_gene_sample_n*_annotation_selection.csv'
        )
    )
    if not selection_candidates:
        raise FileNotFoundError(
            'No representative_gene_sample_n*_annotation_selection.csv files were found under '
            'output/prelim/phase1_metrics_screening_review. Run the screening notebook first or set SELECTION_PATH explicitly.'
        )
    SELECTION_PATH = max(selection_candidates, key=lambda path: path.stat().st_mtime)

api_key = os.environ.get(API_KEY_ENV_VAR)
if not api_key:
    raise RuntimeError(f'Missing AlphaGenome API key. Set {API_KEY_ENV_VAR} in your shell before running this notebook.')

manifest_df = load_loci_manifest(MANIFEST_PATH)
validate_loci_manifest(manifest_df)
selection_df = load_annotation_selection(SELECTION_PATH)
validate_annotation_selection(selection_df, manifest_df)
loci_df = merge_annotation_selection(manifest_df, selection_df)
if LOCUS_IDS is not None:
    loci_df = loci_df[loci_df['locus_id'].isin(set(LOCUS_IDS))].copy()
if loci_df.empty:
    raise ValueError('No loci remain after applying the selection file and any optional locus filters.')
if BATCH_SIZE is not None:
    loci_df['alphagenome_batch_size'] = int(BATCH_SIZE)
if SCORING_MAX_WORKERS is not None:
    loci_df['alphagenome_max_workers'] = int(SCORING_MAX_WORKERS)
if RETRY_WAIT_SECONDS is not None:
    loci_df['alphagenome_retry_wait_seconds'] = int(RETRY_WAIT_SECONDS)

workload_rows = []
for locus_cfg in loci_df.to_dict('records'):
    locus_paths = get_locus_output_paths(PATHS, locus_cfg['locus_id'])
    workload_rows.append(
        estimate_alphagenome_workload(
            locus_cfg,
            PATHS,
            locus_paths,
            max_variants=MAX_VARIANTS,
        )
    )
workload_df = pd.DataFrame(workload_rows)
loci_df = loci_df.merge(
    workload_df[[
        'locus_id',
        'variants_loaded',
        'variants_tss_centered',
        'variants_midpoint_centered',
        'variants_ineligible',
        'variants_to_score',
        'tss_radius_bp',
        'sequence_length_bp',
    ]],
    on='locus_id',
    how='left',
)
total_variants_to_score = int(loci_df['variants_to_score'].fillna(0).sum())

selection_stem = Path(SELECTION_PATH).stem
SUMMARY_PATH = PATHS.output_annotation_alphagenome / f'{selection_stem}_summary.csv'

print(f'Manifest: {MANIFEST_PATH}')
print(f'Selection: {SELECTION_PATH}')
print(f'Summary path: {SUMMARY_PATH}')
print(f'Loci selected for sequential AlphaGenome annotation: {len(loci_df)}')
print(f'Total variants planned for API scoring: {total_variants_to_score}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Max variants per locus: {MAX_VARIANTS}')
print(f'Scoring max workers: {SCORING_MAX_WORKERS}')
print(f'Write batch checkpoints: {WRITE_BATCH_CHECKPOINTS}')
print(f'Retry wait seconds: {RETRY_WAIT_SECONDS}')
display(loci_df[[
    'locus_id',
    'gene_name',
    'gene_id',
    'gtex_tissue',
    'gtex_chrom',
    'variants_tss_centered',
    'variants_midpoint_centered',
    'variants_ineligible',
    'variants_to_score',
]])

In [ ]:
def format_seconds(seconds: float | None) -> str:
    if seconds is None or not pd.notna(seconds):
        return 'n/a'
    total_seconds = int(round(float(seconds)))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f'{hours}h {minutes:02d}m {secs:02d}s'
    if minutes:
        return f'{minutes}m {secs:02d}s'
    return f'{secs}s'


summary_rows = []
run_started_at = datetime.now(timezone.utc)
variants_completed_so_far = 0

for locus_idx, locus_cfg in enumerate(loci_df.to_dict('records'), start=1):
    print(
        f"[{locus_idx}/{len(loci_df)}] {locus_cfg['locus_id']} :: {locus_cfg['gene_name']} "
        f"({locus_cfg['gtex_chrom']}, {locus_cfg['gtex_tissue']}) | planned variants={int(locus_cfg.get('variants_to_score', 0))}"
    )
    row = {
        'locus_id': locus_cfg['locus_id'],
        'gene_name': locus_cfg['gene_name'],
        'gene_id': locus_cfg['gene_id'],
        'gtex_tissue': locus_cfg['gtex_tissue'],
        'gtex_chrom': locus_cfg['gtex_chrom'],
        'status': 'failed',
        'error_message': '',
        'stage': 'annotation',
    }
    started_at = datetime.now(timezone.utc)
    row['started_at'] = started_at.isoformat()
    try:
        locus_paths = get_locus_output_paths(PATHS, locus_cfg['locus_id'])
        if not locus_paths.phase1_alphagenome_ready(locus_cfg['gene_name']):
            raise FileNotFoundError(
                f"AlphaGenome inputs are incomplete for locus {locus_cfg['locus_id']} ({locus_cfg['gene_name']})"
            )
        if locus_paths.annotation_complete(locus_cfg['gene_name']) and not FORCE_RERUN:
            row.update(load_existing_annotation_result(locus_cfg, locus_paths))
            row['status'] = 'skipped_existing'
            print('  skipped_existing')
        else:
            row.update(
                run_alphagenome_annotation(
                    locus_cfg,
                    PATHS,
                    locus_paths,
                    api_key,
                    max_variants=MAX_VARIANTS,
                    write_batch_checkpoints=WRITE_BATCH_CHECKPOINTS,
                    verbose=VERBOSE_PER_BATCH,
                )
            )
            row['status'] = 'completed'
            print(f"  completed: {row['aggregated_variants']} aggregated variants")
    except Exception as exc:  # noqa: BLE001
        row['error_message'] = f'{type(exc).__name__}: {exc}'
        row['traceback'] = traceback.format_exc()
        print(f"  failed: {row['error_message']}")
        if FAIL_FAST:
            finished_at = datetime.now(timezone.utc)
            row['finished_at'] = finished_at.isoformat()
            row['elapsed_seconds'] = round((finished_at - started_at).total_seconds(), 3)
            summary_rows.append(row)
            pd.DataFrame(summary_rows).to_csv(SUMMARY_PATH, index=False)
            raise

    finished_at = datetime.now(timezone.utc)
    row['finished_at'] = finished_at.isoformat()
    row['elapsed_seconds'] = round((finished_at - started_at).total_seconds(), 3)
    summary_rows.append(row)
    locus_variants_done = row.get('variants_attempted')
    if pd.notna(locus_variants_done):
        variants_completed_so_far += int(locus_variants_done)
    elapsed_seconds_total = (finished_at - run_started_at).total_seconds()
    variants_remaining = max(total_variants_to_score - variants_completed_so_far, 0)
    variants_per_second = (
        variants_completed_so_far / elapsed_seconds_total
        if elapsed_seconds_total > 0 and variants_completed_so_far > 0
        else None
    )
    eta_seconds = (
        variants_remaining / variants_per_second
        if variants_per_second not in (None, 0)
        else None
    )
    progress_pct = (
        100.0 * variants_completed_so_far / total_variants_to_score
        if total_variants_to_score > 0
        else float('nan')
    )
    pd.DataFrame(summary_rows).to_csv(SUMMARY_PATH, index=False)
    print(
        f"  overall progress: {variants_completed_so_far}/{total_variants_to_score} variants "
        f"({progress_pct:.1f}%) | elapsed={format_seconds(elapsed_seconds_total)} | "
        f"rate={variants_per_second:.2f} var/s | eta={format_seconds(eta_seconds)}"
        if variants_per_second is not None
        else f"  overall progress: {variants_completed_so_far}/{total_variants_to_score} variants "
        f"({progress_pct:.1f}%) | elapsed={format_seconds(elapsed_seconds_total)} | eta=n/a"
    )
    if SLEEP_SECONDS_BETWEEN_LOCI > 0 and locus_idx < len(loci_df):
        print(f'Sleeping {SLEEP_SECONDS_BETWEEN_LOCI}s before next locus...')
        time.sleep(SLEEP_SECONDS_BETWEEN_LOCI)

summary_df = pd.DataFrame(summary_rows)
summary_display_columns = [
    'locus_id',
    'gene_name',
    'gtex_chrom',
    'status',
    'aggregated_variants',
    'elapsed_seconds',
    'error_message',
]
summary_df = summary_df.reindex(columns=summary_display_columns + [col for col in summary_df.columns if col not in summary_display_columns])
print(f'Wrote: {SUMMARY_PATH}')
display(summary_df[summary_display_columns])